# Atividade 2 -- Tópicos para Computação 1 -- 2026.1

- Escola Superior de Tecnologia
- Profa. Dra. Elloá B. Guedes (ebgcosta@uea.edu.br)
- www.elloaguedes.com
- github.com/elloa
- Data: 10 de março de 2026

## Descrição

A atividade consiste em construir uma rede neural multilayer perceptron para distinguir atributos que definem a renda média de uma pessoa adulta a partir do _UCI Adult Income Dataset_

## Material de Referência para Estudo

- https://docs.pytorch.org/docs/stable/nn.html
- https://machinelearningmastery.com/develop-your-first-neural-network-with-pytorch-step-by-step/
- https://sebastianraschka.com/teaching/pytorch-1h/#7-a-typical-training-loop



## Prazos importantes

- Data de entrega: 16/03/2026
- Modo de entrega: Google Classroom
- Estratégia de desenvolvimento: Trios

#### ALUNOS
Caio Jorge da Cunha Queiroz - 2315310028\
Lucas Maciel Gomes - 2315310014\
Izabella de Lima Catrinck - 2315310033

# Abertura e Análise Exploratória do Dataset

- A tarefa a ser considerada é uma tarefa de classificação binária, cujo objetivo é determinar se a renda média de uma pessoa adulta excedará os US $50,000 por ano.
- O dataset em questão pode ser obtido aqui: https://archive.ics.uci.edu/dataset/2/adult
- Faça uma pré-seleção dos exemplos, conforme sugerido pelos autores: (AAGE>16) && (AGI>100) && (AFNLWGT>1) && (HRSWK>0))
- Exclua todas as linhas com dados faltantes
- Quantos exemplos viáveis há no dataset?
- Preparação de atributos: todos os atributos categóricos devem ser codificados com One-Hot Encoding
- O dataset é balanceado?

In [ ]:
import numpy as np
import polars as pl
import torch
from polars import DataFrame, Series
from torch import Tensor

In [ ]:
colunas: list[str] = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "income",
]

df_train: DataFrame = pl.read_csv("adult/adult.data", has_header=False, new_columns=colunas)
df_train.head()


age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str
39,""" State-gov""",""" 77516""",""" Bachelors""",""" 13""",""" Never-married""",""" Adm-clerical""",""" Not-in-family""",""" White""",""" Male""",""" 2174""",""" 0""",""" 40""",""" United-States""",""" <=50K"""
50,""" Self-emp-not-inc""",""" 83311""",""" Bachelors""",""" 13""",""" Married-civ-spouse""",""" Exec-managerial""",""" Husband""",""" White""",""" Male""",""" 0""",""" 0""",""" 13""",""" United-States""",""" <=50K"""
38,""" Private""",""" 215646""",""" HS-grad""",""" 9""",""" Divorced""",""" Handlers-cleaners""",""" Not-in-family""",""" White""",""" Male""",""" 0""",""" 0""",""" 40""",""" United-States""",""" <=50K"""
53,""" Private""",""" 234721""",""" 11th""",""" 7""",""" Married-civ-spouse""",""" Handlers-cleaners""",""" Husband""",""" Black""",""" Male""",""" 0""",""" 0""",""" 40""",""" United-States""",""" <=50K"""
28,""" Private""",""" 338409""",""" Bachelors""",""" 13""",""" Married-civ-spouse""",""" Prof-specialty""",""" Wife""",""" Black""",""" Female""",""" 0""",""" 0""",""" 40""",""" Cuba""",""" <=50K"""


In [ ]:
df_train: DataFrame = df_train.with_columns(
    pl.col("income").str.strip_chars())

df_train: DataFrame = df_train.with_columns(
    (pl.col("income") == ">50K").cast(pl.Int8).alias("income"))

df_train.head()

age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
i64,str,str,str,str,str,str,str,str,str,str,str,str,str,i8
39,""" State-gov""",""" 77516""",""" Bachelors""",""" 13""",""" Never-married""",""" Adm-clerical""",""" Not-in-family""",""" White""",""" Male""",""" 2174""",""" 0""",""" 40""",""" United-States""",0
50,""" Self-emp-not-inc""",""" 83311""",""" Bachelors""",""" 13""",""" Married-civ-spouse""",""" Exec-managerial""",""" Husband""",""" White""",""" Male""",""" 0""",""" 0""",""" 13""",""" United-States""",0
38,""" Private""",""" 215646""",""" HS-grad""",""" 9""",""" Divorced""",""" Handlers-cleaners""",""" Not-in-family""",""" White""",""" Male""",""" 0""",""" 0""",""" 40""",""" United-States""",0
53,""" Private""",""" 234721""",""" 11th""",""" 7""",""" Married-civ-spouse""",""" Handlers-cleaners""",""" Husband""",""" Black""",""" Male""",""" 0""",""" 0""",""" 40""",""" United-States""",0
28,""" Private""",""" 338409""",""" Bachelors""",""" 13""",""" Married-civ-spouse""",""" Prof-specialty""",""" Wife""",""" Black""",""" Female""",""" 0""",""" 0""",""" 40""",""" Cuba""",0


### Respostas da primeira seção - Abertura e Análise Exploratória

**Quantos exemplos viáveis há no dataset?**

Partimos de um total de 48.842 exemplos (32.561 do arquivo de treino + 16.281 do arquivo de teste). Após a aplicação do filtro dos autores `(age > 16) && (capital-gain > 100) && (fnlwgt > 1) && (hours-per-week > 0)`, restaram 4.035 exemplos. A maior redução foi causada pelo critério `capital-gain > 100`, já que a grande maioria das pessoas no censo possui ganho de capital igual a zero. Em seguida, removemos 245 linhas que continham valores faltantes (`'?'`), concentrados nas colunas `workclass`, `occupation` e `native-country`. Ao final, obtivemos 3.790 exemplos viáveis.

**Preparação dos atributos**

Das 15 colunas originais, 8 são categóricas (`workclass`, `education`, `marital-status`, `occupation`, `relationship`, `race`, `sex`, `native-country`). Após aplicarmos o One-Hot Encoding nessas colunas e codificarmos o atributo-alvo como binário (<=50K → 0, >50K → 1), o dataset passou de 15 colunas para 102 colunas, resultando em 101 atributos preditores.

**O dataset é balanceado?**

Não, o dataset não é balanceado. A distribuição das classes ficou:

| Classe | Quantidade | Proporção |
|:-:|:-:|:-:|
| >50K | 2.375 | 62,66% |
| ≤50K | 1.415 | 37,34% |

É interessante notar que essa proporção é o inverso do que ocorre no dataset completo (onde ~76% é ≤50K). Essa inversão acontece porque o filtro `capital-gain > 100` seleciona pessoas com ganhos de capital significativos.

## Preparação dos exemplos para treino e teste

- Separe os atributos preditores do atributo-alvo
- Faça uma partição do tipo holdout 70/30 de forma aleatória, use seed = 42
- Codifique os exemplos em tensores pytorch

In [ ]:
categorical: list[str] = df_train.select(pl.col(pl.Utf8)).columns

df: DataFrame = df_train.to_dummies(columns=categorical)

df.head()

age,workclass_ ?,workclass_ Federal-gov,workclass_ Local-gov,workclass_ Never-worked,workclass_ Private,workclass_ Self-emp-inc,workclass_ Self-emp-not-inc,workclass_ State-gov,workclass_ Without-pay,workclass_null,fnlwgt_ 100009,fnlwgt_ 100029,fnlwgt_ 100054,fnlwgt_ 100063,fnlwgt_ 100067,fnlwgt_ 100079,fnlwgt_ 100099,fnlwgt_ 100109,fnlwgt_ 100135,fnlwgt_ 100147,fnlwgt_ 100154,fnlwgt_ 100168,fnlwgt_ 100188,fnlwgt_ 100219,fnlwgt_ 100226,fnlwgt_ 100228,fnlwgt_ 100252,fnlwgt_ 100270,fnlwgt_ 100292,fnlwgt_ 100295,fnlwgt_ 100313,fnlwgt_ 100316,fnlwgt_ 100321,fnlwgt_ 100345,fnlwgt_ 100375,fnlwgt_ 100405,…,native-country_ Ecuador,native-country_ El-Salvador,native-country_ England,native-country_ France,native-country_ Germany,native-country_ Greece,native-country_ Guatemala,native-country_ Haiti,native-country_ Holand-Netherlands,native-country_ Honduras,native-country_ Hong,native-country_ Hungary,native-country_ India,native-country_ Iran,native-country_ Ireland,native-country_ Italy,native-country_ Jamaica,native-country_ Japan,native-country_ Laos,native-country_ Mexico,native-country_ Nicaragua,native-country_ Outlying-US(Guam-USVI-etc),native-country_ Peru,native-country_ Philippines,native-country_ Poland,native-country_ Portugal,native-country_ Puerto-Rico,native-country_ Scotland,native-country_ South,native-country_ Taiwan,native-country_ Thailand,native-country_ Trinadad&Tobago,native-country_ United-States,native-country_ Vietnam,native-country_ Yugoslavia,native-country_null,income
i64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,…,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,i8
39,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
50,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
38,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
53,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
28,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
X: DataFrame = df.drop("income")
y: Series = df["income"]

In [ ]:
np.random.seed(42)

indices = np.random.permutation(len(df))
split = int(len(df) * 0.7)

train_idx = indices[:split]
test_idx = indices[split:]

X_train: DataFrame = X[train_idx]
X_test: DataFrame = X[test_idx]

y_train: Series = y[train_idx]
y_test: Series = y[test_idx]

In [ ]:
X_train_t: Tensor = torch.tensor(X_train.to_numpy(), dtype=torch.float32)
X_test_t: Tensor = torch.tensor(X_test.to_numpy(), dtype=torch.float32)

y_train_t: Tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).view(-1,1)
y_test_t: Tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32).view(-1,1)

input_size: int = X_train_t.shape[1]

## Rede MLP de Camada Única

- Proponha uma rede neural multilayer perceptron no Pytorch cujo número de neurônios dá-se como segue:
    - Camada de entrada: quantidade de atributos preditores
    - Camada oculta: 100 neurônios (Linear)
    - Função de ativação: ReLU
    - Camada de saída: 1 neurônio com função de ativação sigmoidal (usar degrau em 0.5)
    - Função de Perda: Entropia cruzada binária
- Hiperparâmetros:
   - Épocas: 100
   - Taxa de aprendizado: 10^-4
- Treine a rede com a partição reservada para essa finalidade
    - Otimizador: Mini-batch SGD com batch size de 16
- Mostre o gráfico da função de perda ao longo das épocas

### Avaliação da MLP de Camada Única

- Obtenha as métricas de desempenho para a partição de testes
  1. Acurácia balanceada
  2. Precisão balanceada
  3. Revocação balanceada
  4. F1-Score balanceado
- Imprima a matriz de confusão do teste
- Avalie de forma crítica: Como foi o desempenho da rede perante essa tarefa?

## Rede MLP com duas camadas ocultas

- Proponha uma rede neural multilayer perceptron no Pytorch cujo número de neurônios dá-se como segue:
    - Camada de entrada: quantidade de atributos preditores
    - Camada oculta 1: 100 neurônios (Linear)
    - Função de ativação: ReLU
    - Camada oculta 2: 50 neurônios (Linear)
    - Função de ativação: ReLU
    - Camada de saída: 1 neurônio com função de ativação sigmoidal (usar degrau em 0.5)
    - Função de Perda: Entropia cruzada binária
- Hiperparâmetros:
   - Épocas: 100
   - Taxa de aprendizado: 10^-4
- Treine a rede com a partição reservada para essa finalidade
    - Otimizador: Mini-batch SGD com batch size de 16
- Mostre o gráfico da função de perda ao longo das épocas

### Avaliação da MLP com Duas Camadas Ocultas

- Obtenha as métricas de desempenho para a partição de testes
  1. Acurácia balanceada
  2. Precisão balanceada
  3. Revocação balanceada
  4. F1-Score balanceado
- Imprima a matriz de confusão do teste
- Avalie de forma crítica: Como foi o desempenho da nova rede perante essa tarefa? Houve melhora?

## Sua rede MLP

- Proponha uma rede neural multilayer perceptron no Pytorch com uma ou duas camadas ocultas e número de neurônios conforme sua livre escolha, levando em conta razoabilidade,  tempo de execução no hardware disponível e prazo de entrega da atividade
- Hiperparâmetros:
   - Modifique-os como desejar
- Treine a rede com a partição reservada para essa finalidade
    - Otimizador: Mini-batch SGD com batch size de 16
- Mostre o gráfico da função de perda ao longo das épocas

### Avaliação da sua rede MLP

- Obtenha as métricas de desempenho para a partição de testes
  1. Acurácia balanceada
  2. Precisão balanceada
  3. Revocação balanceada
  4. F1-Score balanceado
- Imprima a matriz de confusão do teste

## Análise Comparativa Final

Construa uma tabela (pacote prettytable) com as métricas de desempenho das três redes propostas na partição de testes e justifique, com base em argumentos de performance, eficiência e aderência ao problema, qual delas possivelmente obteve melhor desempenho na tarefa.